# FleetIQ — NYC TLC Data Quality Assessment

## Objective
This notebook converts the audit into a small, explicit cleaning plan for completed-trip duration prediction. It counts each rule, applies only the necessary drop rules, and validates the result. It never overwrites the raw source.

**Input:** one raw NYC Yellow Taxi file.  
**Output:** a transparent rule summary and a cleaned in-memory table ready for the next project stage.


## Notebook Audit
The original DQA was cautious but used a generic rule engine, issue register, severity scorecard, and typed helper functions. They made the result look sophisticated while hiding the core idea: each cleaning rule is simply a True/False condition over rows.

| Part | Level | Decision |
| --- | --- | --- |
| Boolean masks and `.loc` filtering | 🟢 | Keep. |
| Combining masks with `|` | 🟡 | Keep and practise. |
| Generic rule/score framework | 🔴 | Remove for now. |

## Minimum Viable Notebook
`Raw trips → temporary target → named masks → count each rule → combine DROP masks → cleaned copy → validate`

An outlier is unusual, not automatically invalid. We drop only records that cannot provide a valid target or essential zone context.


## Concepts You Need to Understand
- A **boolean mask** contains one True/False value per row.
- `|` means “or”: it creates the union of failures and avoids double-counting.
- **DROP** means the row cannot support the current modelling objective.
- **FLAG** means keep the row and investigate it later.
- Post-cleaning **assertions** verify the rules worked instead of trusting the code blindly.


## 1. Load Raw Data

### What are we doing?
Locate and load one raw trip file.

### Why are we doing it?
We first learn rules on one month, then repeat approved rules for all months.

### What should I expect?
The chosen filename and original row count.


In [1]:
from pathlib import Path
import pandas as pd

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / "data" / "raw").exists():
            return folder
    return current

RAW_DATA_DIR = find_project_root() / "data" / "raw" / "nyc_tlc"
parquet_files = sorted(RAW_DATA_DIR.glob("yellow_tripdata_*.parquet"))
if not parquet_files:
    raise FileNotFoundError("Add Yellow Taxi Parquet files to data/raw/nyc_tlc.")

trip_file = parquet_files[0]
raw_trips = pd.read_parquet(trip_file)
print(f"Loaded {trip_file.name}: {len(raw_trips):,} rows")


Loaded yellow_tripdata_2025-01.parquet: 3,475,226 rows


### What happened? What should I check?
Keep `raw_trips` unchanged. A separate `cleaned_trips` table later makes it clear which data is raw and which data is derived.


## 2. Make the Candidate Target

### What are we doing?
Parse timestamps and calculate duration in minutes.

### Why are we doing it?
This is both the model target and the basis for an impossible-duration rule.

### What should I expect?
Summary statistics for a temporary Series; raw data stays unchanged.


In [2]:
pickup = pd.to_datetime(raw_trips["tpep_pickup_datetime"], errors="coerce")
dropoff = pd.to_datetime(raw_trips["tpep_dropoff_datetime"], errors="coerce")
duration_minutes = (dropoff - pickup).dt.total_seconds() / 60
display(duration_minutes.describe(percentiles=[0.01, 0.5, 0.95, 0.99]))


count    3.475226e+06
mean     1.501812e+01
std      3.871358e+01
min     -5.147232e+04
1%       6.666667e-01
50%      1.170000e+01
95%      3.620000e+01
99%      5.891667e+01
max      5.626317e+03
dtype: float64

## 3. Write Explicit Cleaning Rules

### What are we doing?
Give each condition a descriptive name, then count it.

### Why are we doing it?
Each line is a policy you should be able to defend. We do not hide policy in an abstract framework.

### What should I expect?
A rule table. The same row can be counted by more than one individual rule.


In [3]:
# DROP: a target cannot be constructed without timestamps or with non-positive duration.
missing_timestamp = pickup.isna() | dropoff.isna()
non_positive_duration = duration_minutes.isna() | (duration_minutes <= 0)

# DROP: planned zone-level features need both location IDs.
missing_pickup_zone = raw_trips["PULocationID"].isna()
missing_dropoff_zone = raw_trips["DOLocationID"].isna()

# FLAG ONLY: unusual is not the same as invalid.
negative_distance = raw_trips["trip_distance"] < 0
zero_distance = raw_trips["trip_distance"] == 0
long_duration = duration_minutes > duration_minutes.quantile(0.99)

rule_summary = pd.DataFrame({
    "rule": ["missing timestamp", "non-positive duration", "missing pickup zone", "missing drop-off zone", "negative distance", "zero distance", "top 1% duration"],
    "action": ["DROP", "DROP", "DROP", "DROP", "FLAG", "FLAG", "FLAG"],
    "affected_rows": [missing_timestamp.sum(), non_positive_duration.sum(), missing_pickup_zone.sum(), missing_dropoff_zone.sum(), negative_distance.sum(), zero_distance.sum(), long_duration.sum()]
})
rule_summary["affected_percent"] = (rule_summary["affected_rows"] / len(raw_trips) * 100).round(3)
display(rule_summary)


,rule,action,affected_rows,affected_percent
0,missing timestamp,DROP,0,0.000
1,non-positive duration,DROP,2051,0.059
2,missing pickup zone,DROP,0,0.000
3,missing drop-off zone,DROP,0,0.000
4,negative distance,FLAG,0,0.000
5,zero distance,FLAG,90893,2.615
6,top 1% duration,FLAG,34725,0.999


### What happened? What should I check?
Do not add individual row counts to find total removals. A row with a missing timestamp may also have a missing zone. Combine DROP masks first.


## 4. Apply Only the Required DROP Rules

### What are we doing?
Create one union mask, keep the remaining rows, and then add the derived target.

### Why are we doing it?
This removes only rows that cannot support the present objective. It retains rare but potentially valid trips.

### What should I expect?
An auditable before/after count and a new DataFrame.


In [4]:
drop_row = (
    missing_timestamp
    | non_positive_duration
    | missing_pickup_zone
    | missing_dropoff_zone
)

cleaned_trips = raw_trips.loc[~drop_row].copy()
cleaned_trips["trip_duration_minutes"] = duration_minutes.loc[~drop_row]

print(f"Rows before cleaning: {len(raw_trips):,}")
print(f"Rows removed:        {drop_row.sum():,}")
print(f"Rows remaining:      {len(cleaned_trips):,}")


Rows before cleaning: 3,475,226
Rows removed:        2,051
Rows remaining:      3,473,175


### What happened? What should I check?
`.copy()` creates a distinct cleaned table. The target column is a derived value, so it belongs in cleaned data rather than raw input. This is an in-memory preview; a later notebook can save an approved processed dataset.


## 5. Validate the Result

### What are we doing?
Rerun the conditions that the DROP policy promised to fix.

### Why are we doing it?
Cleaning is not trustworthy unless postconditions are checked.

### What should I expect?
All assertions pass silently, followed by a confirmation message.


In [5]:
assert cleaned_trips["tpep_pickup_datetime"].notna().all()
assert cleaned_trips["tpep_dropoff_datetime"].notna().all()
assert cleaned_trips["PULocationID"].notna().all()
assert cleaned_trips["DOLocationID"].notna().all()
assert (cleaned_trips["trip_duration_minutes"] > 0).all()
print("Validation passed: retained rows have a positive target and both zone IDs.")


Validation passed: retained rows have a positive target and both zone IDs.


## Decisions We Postpone
- Zero distance can be a short ride or a data issue: keep and investigate it with duration.
- Top 1% duration is a flag, not an automatic deletion rule.
- Fare and total amount are likely known after the trip, so they risk target leakage for a pre-trip estimate.
- Passenger-count treatment needs its own missingness/domain decision.

This keeps results scientifically honest: unusual values are visible, and future information does not enter an ETA prediction.


## New Notebook Architecture
`Raw trips → temporary target → named masks → rule counts → union of DROP masks → cleaned copy → validation`

## Prerequisites
pandas boolean masks, `|`, `.loc`, `.copy()`, datetime subtraction, and leakage.

## Try It Yourself
1. Add a FLAG for passenger counts above 6.
2. Show the five longest retained trips.
3. Compare removal percentage for a different month.
4. Add a missing-distance rule and decide whether it should DROP or FLAG.

## Solution Hints
Use `raw_trips["passenger_count"] > 6`, `.nlargest(5, "trip_duration_minutes")`, and `drop_row.mean() * 100`.

## Can I Rebuild This?
- [ ] Load raw data without changing it
- [ ] Calculate a duration target
- [ ] Name one boolean mask per rule
- [ ] Explain DROP versus FLAG
- [ ] Combine DROP masks with `|`
- [ ] Create a copied cleaned DataFrame
- [ ] Add the target after filtering
- [ ] Validate the postconditions

## Knowledge Boundary
### 🔴 I MUST UNDERSTAND
Boolean masks, filtering, target creation, DROP versus FLAG, leakage, and assertions.
### 🟡 I SHOULD UNDERSTAND CONCEPTUALLY
Percentiles, outlier treatment, and how cleaning affects modelling.
### 🟢 AI / LIBRARY CAN HELP ME WITH
Exact pandas syntax, formatted tables, and TLC documentation lookup.

## Next Step
Repeat the approved rules across your selected months, save a versioned processed dataset with before/after counts, then start feature engineering and a temporal split.
